# Project: Neutrino Portal at the Muon Collider

## Load Libraries

In [1]:
import os
import pandas as pd
import numpy as np
import pathlib
import ast
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import json

## Common Analysis Functions

In [2]:
def fix_dataframe(data):

    # Initialize the column 'ievent' with zeros
    data['ievent'] = 0
    # Variable to keep track of the increment for column 'ievent'
    counter = 0
    # Loop through the DataFrame rows
    for index, row in data.iterrows():
        if row['iparticle'] == 0: counter += 1
        data.at[index, 'ievent'] = counter
    #return
    return data

def calculate_observables(data):
    events = []
    grouped_data = data.groupby('ievent')

    for ievent, evt in grouped_data:
        # Reset or initialize variables for each event
        e_mu_minus, e_mu_plus, has_charm, e_em, e_visible, ptx, pty, ht = 0, 0, 0, 0, 0, 0, 0, 0
        pt_mu_minus, pt_mu_minus, phi_mu_minus, phi_mu_minus = 0, 0, 0, 0
        
        # Iterate over particles in the event
        for _, row in evt.iterrows():
            truth_energy, pid, parent_pid1 = row.iloc[2], row.iloc[3], row.iloc[8]
            px, py, pz, e  =  row.iloc[4], row.iloc[5], row.iloc[6], row.iloc[7]
               
            # Check for charm particles
            if abs(parent_pid1) in {411, 421, 431, 4122,15,521,511,531,443,5122,4232,4112,5232}: 
                has_charm = 1
            
            # Update energies for mu- and mu+
            if pid == 13: 
                if e>e_mu_minus:
                    e_mu_minus = e
                    pt_mu_minus = np.sqrt(px**2 + py**2) 
                    phi_mu_minus = np.arctan2(py, px)
            elif pid == -13: 
                if e>e_mu_plus:
                    e_mu_plus = e
                    pt_mu_plus = np.sqrt(px**2 + py**2) 
                    phi_mu_plus = np.arctan2(py, px)
            
            # Sum EM and visible energy
            if abs(pid) in {22, 11}: 
                e_em += e
            if abs(pid) not in {12, 14, 16, 39}: 
                e_visible += e
                ptx += px
                pty += py
                ht += np.sqrt(px**2 + py**2)
        
        # Calculate missing pT
        pt_mis = np.sqrt(ptx**2 + pty**2)
        phi_mis = np.arctan2(pty, ptx)
        
        # Append event observables
        events.append([ievent, truth_energy, e_mu_minus, e_mu_plus, e_em, has_charm, e_visible, pt_mis, ht, pt_mu_minus, pt_mu_plus, phi_mu_minus, phi_mu_plus, phi_mis])

    # Construct DataFrame from results
    columns = ['ievent', 'truth_energy', 'e_mu_minus', 'e_mu_plus', 'e_em', 'has_charm', 'e_visible', 'pt_mis', 'ht', 'pt_mu_minus', 'pt_mu_plus', 'phi_mu_minus', 'phi_mu_plus', 'phi_mis']
    observables = pd.DataFrame(events, columns=columns)
    
    return observables


## Calculate Observables

-  read events from one file

In [3]:
data = pd.read_csv("output_events_NC_numu/events_22.53.csv.zip")
data

,ievent,iparticle,truth_energy,pid,px,py,pz,e,parent_pid1,parent_pid2
0,NaN,0,22.53,14,-0.934,-0.835,7.702,7.803,14,14
1,NaN,1,22.53,211,-0.282,-0.034,1.208,1.249,4,2203
2,NaN,2,22.53,-211,1.316,-0.109,2.203,2.572,-411,90
3,NaN,3,22.53,-13,-0.311,0.066,0.355,0.488,4122,90
4,NaN,4,22.53,14,0.223,0.523,4.047,4.087,4122,90
...,...,...,...,...,...,...,...,...,...,...
27676,NaN,4,22.53,-14,-0.884,-0.032,2.474,2.627,-421,90
27677,NaN,5,22.53,321,0.044,-0.085,0.677,0.843,-421,90
27678,NaN,6,22.53,-13,0.262,0.260,3.125,3.149,411,90
27679,NaN,7,22.53,14,-0.253,-0.503,3.075,3.126,411,90


- fix missing ievent entries

In [4]:
data = fix_dataframe(data)
data

,ievent,iparticle,truth_energy,pid,px,py,pz,e,parent_pid1,parent_pid2
0,1,0,22.53,14,-0.934,-0.835,7.702,7.803,14,14
1,1,1,22.53,211,-0.282,-0.034,1.208,1.249,4,2203
2,1,2,22.53,-211,1.316,-0.109,2.203,2.572,-411,90
3,1,3,22.53,-13,-0.311,0.066,0.355,0.488,4122,90
4,1,4,22.53,14,0.223,0.523,4.047,4.087,4122,90
...,...,...,...,...,...,...,...,...,...,...
27676,2368,4,22.53,-14,-0.884,-0.032,2.474,2.627,-421,90
27677,2368,5,22.53,321,0.044,-0.085,0.677,0.843,-421,90
27678,2368,6,22.53,-13,0.262,0.260,3.125,3.149,411,90
27679,2368,7,22.53,14,-0.253,-0.503,3.075,3.126,411,90


- evaulates all necessary obseravbles for each event

In [5]:
observables = calculate_observables(data)
observables

,ievent,truth_energy,e_mu_minus,e_mu_plus,e_em,has_charm,e_visible,pt_mis,ht,pt_mu_minus,pt_mu_plus,phi_mu_minus,phi_mu_plus,phi_mis
0,1,22.53,0.000,0.488,0.000,1,11.622,0.775126,2.910989,0.000000,0.317926,0.000000,2.932476,0.412854
1,2,22.53,1.261,0.324,6.450,0,21.574,1.154859,2.556138,0.160730,0.022472,1.154467,0.363979,1.149759
2,3,22.53,0.853,4.620,1.724,1,18.463,1.002941,2.973211,0.249163,0.216670,-3.105464,0.837638,2.851377
3,4,22.53,1.275,0.539,1.218,0,4.188,1.296556,1.546107,0.497315,0.240468,1.411267,1.508378,1.426063
4,5,22.53,0.534,2.197,0.000,1,16.143,1.784818,2.795638,0.257783,0.943064,2.970067,-0.422854,0.302083
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2363,2364,22.53,0.000,1.484,2.573,1,19.545,1.125340,3.060091,0.000000,0.577541,0.000000,-1.527496,-2.754321
2364,2365,22.53,0.135,0.250,0.398,0,3.141,0.623046,0.826662,0.058034,0.122409,3.107124,-1.652581,-2.772114
2365,2366,22.53,1.190,1.634,1.204,1,20.523,0.915495,2.749552,0.163466,0.125929,-2.395136,-2.925508,-1.275992
2366,2367,22.53,0.000,4.545,4.098,1,15.692,1.280407,2.987993,0.000000,0.366947,0.000000,-1.274902,0.321010


- let us systematically calculate obvservables for all files

In [ ]:
#define energies
energies = [
    14.21, 17.90, 22.53, 28.37, 35.71, 44.964, 56.60, 71.26, 89.71, 
    112.94, 142.19, 179.00, 225.35, 283.70, 357.16, 449.64, 566.07, 712.64, 897.16,
    1129.46, 1421.90,
] 

#define processes
processes_bg = ["CC_numu", "CC_nuebar", "NC_numu", "NC_nuebar"]
processes_sig = ["0.125GeV","0.5GeV","1GeV","3.16GeV","7.94GeV","15.84GeV","19.95GeV"]
processes = processes_bg + processes_sig

# Loop through all energies
for process in processes: 
    for energy in energies:
        filename = f"output_events_{process}/events_{str(energy)}.csv.zip"
        
        # Check if the file exists
        if not os.path.isfile(filename):
            print(f"File not found: {filename}")
            continue

        print(process, energy)
        data = pd.read_csv(filename)
        
        # Check if DataFrame is empty
        if data.empty:
            print(f"Data is empty for {filename}")
            continue
        
        # Check if 'ievent' column exists
        if 'ievent' not in data.columns:
            print(f"Column 'ievent' not found in {filename}")
            continue
        
        # Check if the first row has a valid value
        if 0 not in data.index or np.isnan(data.loc[0, 'ievent']):
            print(f"Fixing DataFrame for {filename}")
            data = fix_dataframe(data)
        
        # Calculate observables and save
        observables = calculate_observables(data)
        csv_file = f"output_events_{process}/observables_{str(energy)}.csv.zip"
        observables.to_csv(csv_file, index=False, compression='zip')


CC_numu 14.21
CC_numu 17.9
CC_numu 22.53
CC_numu 28.37
CC_numu 35.71
CC_numu 44.964
CC_numu 56.6
CC_numu 71.26
CC_numu 89.71
CC_numu 112.94
CC_numu 142.19
CC_numu 179.0
CC_numu 225.35
CC_numu 283.7
CC_numu 357.16
CC_numu 449.64
CC_numu 566.07
CC_numu 712.64
CC_numu 897.16
CC_numu 1129.46
CC_numu 1421.9
CC_nuebar 14.21
Fixing DataFrame for output_events_CC_nuebar/events_14.21.csv.zip
CC_nuebar 17.9
Fixing DataFrame for output_events_CC_nuebar/events_17.9.csv.zip
CC_nuebar 22.53
Fixing DataFrame for output_events_CC_nuebar/events_22.53.csv.zip
CC_nuebar 28.37
CC_nuebar 35.71
Fixing DataFrame for output_events_CC_nuebar/events_35.71.csv.zip
CC_nuebar 44.964
Fixing DataFrame for output_events_CC_nuebar/events_44.964.csv.zip
CC_nuebar 56.6
CC_nuebar 71.26
Fixing DataFrame for output_events_CC_nuebar/events_71.26.csv.zip
CC_nuebar 89.71
Fixing DataFrame for output_events_CC_nuebar/events_89.71.csv.zip
CC_nuebar 112.94
Fixing DataFrame for output_events_CC_nuebar/events_112.94.csv.zip
CC_nueb